# 9 - Agentic Systems and What to Test

An agent is not a single answer but a **trajectory**: a sequence of tool calls, gate decisions and an outcome. The framework scores it at three levels, independently, so a run can pass one and fail another:

- **outcome** --- did the final answer / labeled decisions match ground truth;
- **trajectory structure** --- did the workflow run in order and escalate by the right path;
- **process health** --- step count, tool failures, clean termination.

## The system under test is an interface
A SUT is any `(query, context) -> answer` callable. Returning a `SUTResult` (answer, per-component predictions, trajectory, status) unlocks the trajectory and component levels. In production the SUT wraps the real agent --- `apps/complaint_sut.AgentSUT` maps a live `build_complaint_harness` trajectory to a `SUTResult`. Here we use a small deterministic stand-in so the notebook runs without a GPU; the scoring API is identical.

In [ ]:
import sys, pathlib
import proofloop.suite as gt
cat = gt.Catalog.load()
print(cat.summary())

In [ ]:
from proofloop.suite.evaluate import SUTResult, run, summary, weak_link
workflow = ['classify', 'extract', 'search', 'flag', 'draft']

def sut(query, context=''):                      # stand-in for AgentSUT
    bad = '[clarity=Misleading]' in query        # this SUT slips on misleading framing
    return SUTResult(answer='OK' if not bad else 'WRONG',
                     components={'classification': 'complaint' if not bad else 'other'},
                     trajectory=[{'tool': t, 'ok': True} for t in workflow],
                     status='ok' if not bad else 'failed')

`evaluate.run` executes each scenario through the SUT and returns one row per scenario; `summary` aggregates outcome accuracy and workflow adherence.

In [ ]:
items = [gt.QAItem(qid=f's{i}', query='[clarity=Clear] msg', answer='OK',
                   components={'expected_classification': 'complaint',
                               'expected_escalation': False}) for i in range(3)]
suite = gt.resolve(cat, ['exact_recall'], ['clarity'], mode='cross')
scns = gt.compose(suite, items, n_runs=2, seed=1)
rows = run(scns, sut, workflow_order=workflow)
print(summary(rows))

## Self-check

In [ ]:
assert summary(rows)['workflow_adherence'] == 1.0      # ran in order every time
assert 'accuracy' in summary(rows)
print('OK - agentic: what to test')